In [6]:
import pandas as pd
from datetime import date
from pathlib import Path

In [7]:
def download_viirs_snpp_peru(year: int, save_path: str | None = None) -> tuple[pd.DataFrame, str]:
    """
    Download NASA FIRMS per-country VIIRS-SNPP fires for Peru for a given year.

    Parameters
    ----------
    year : int
        Year to fetch (VIIRS-SNPP is available from ~2012 onward).
    save_path : str | None
        If provided, saves the downloaded CSV to this path.

    Returns
    -------
    (df, url) : (pandas.DataFrame, str)
        The DataFrame with the fires and the exact URL used.

    Example
    -------
    df, url = download_viirs_snpp_peru(2013, "viirs-snpp_2013_Peru.csv")
    """
    # Basic sanity checks
    current_year = date.today().year
    if not isinstance(year, int):
        raise TypeError("year must be an integer, e.g., 2013")
    if year < 2012 or year > current_year:
        raise ValueError(f"year must be between 2012 and {current_year}")

    # URL pattern from your example
    url = f"https://firms.modaps.eosdis.nasa.gov/data/country/viirs-snpp/{year}/viirs-snpp_{year}_Peru.csv"

    # Read directly with pandas
    df = pd.read_csv(url)

    return( df )


In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from typing import Literal, Optional, Dict, List


def fires_clean(
    df: pd.DataFrame,
    lat_col: str = "latitude",
    lon_col: str = "longitude",
    acq_time_col: str = "acq_time",
    acq_date_col: str = "acq_date",
    crs: str = "EPSG:4326",
) -> gpd.GeoDataFrame:
    """
    Convert raw VIIRS fires DataFrame into a GeoDataFrame and add year/month.

    Assumptions
    ----------
    - The input has columns:
      ['latitude','longitude','bright_ti4','scan','track','acq_date','acq_time',
       'satellite','instrument','confidence','version','bright_ti5','frp','daynight','type']
    - `acq_time` is a DATE string in the form 'YYYY-MM-DD' (per your example).
      If not parseable, we fall back to `acq_date`.

    Returns
    -------
    gdf : GeoDataFrame (EPSG:4326)
        Adds columns: 'event_dt' (Timestamp), 'year' (int), 'month' (int)
    """
    df = df.copy()

    # Coerce coordinates and drop invalid rows
    df[lat_col] = pd.to_numeric(df[lat_col], errors="coerce")
    df[lon_col] = pd.to_numeric(df[lon_col], errors="coerce")
    df = df.dropna(subset=[lat_col, lon_col])

    # Parse datetime (prefer acq_time as per your format; fallback to acq_date)
    dt = pd.to_datetime(df[acq_time_col], errors="coerce", utc=False, infer_datetime_format=True)
    if dt.isna().all() and acq_date_col in df.columns:
        dt = pd.to_datetime(df[acq_date_col], errors="coerce", utc=False, infer_datetime_format=True)

    if dt.isna().all():
        raise ValueError("Could not parse dates from 'acq_time' or 'acq_date'.")

    df["event_dt"] = dt
    df["year"] = df["event_dt"].dt.year.astype("Int64")
    df["month"] = df["event_dt"].dt.month.astype("Int64")

    # Build geometry
    geom = [Point(xy) for xy in zip(df[lon_col], df[lat_col])]
    gdf = gpd.GeoDataFrame(df, geometry=geom, crs=crs)

    return gdf


In [39]:


def intersect_and_collapse_fires_peru(
    fires_gdf: gpd.GeoDataFrame,
    peru_lvl2_gdf: gpd.GeoDataFrame,
    id_col: str = "prov_id",
    level: Literal["year", "year_month"] = "year",
    agg_extra: Optional[Dict[str, str]] = None,
    keep_province_attrs: Optional[List[str]] = None,
) -> pd.DataFrame:
    """
    Intersect fire points with Peru Level-2 polygons and collapse counts.

    Parameters
    ----------
    fires_gdf : GeoDataFrame
        Output of `fires_clean` (must contain 'year' and 'month' and a geometry).
    peru_lvl2_gdf : GeoDataFrame
        Peru polygons (e.g., GADM level-2). Must contain an ID column (default 'prov_id').
    id_col : str
        Province identifier column name in `peru_lvl2_gdf`.
    level : 'year' or 'year_month'
        Aggregation level. Default 'year'. If 'year_month', groups by (year, month).
    agg_extra : dict, optional
        Extra aggregations on numeric fields, e.g., {'frp': 'sum'} to sum FRP.
    keep_province_attrs : list[str], optional
        Extra province columns (e.g., ['province','region']) to include in the output.

    Returns
    -------
    df_out : pandas.DataFrame
        Columns include: [id_col], 'n_fires', and time keys ('year' [, 'month']).
        If `keep_province_attrs` provided, they are included (deduplicated merge).
    """
    if fires_gdf.empty:
        # Build empty skeleton with available province ids & time keys if desired
        base = peru_lvl2_gdf[[id_col]].drop_duplicates().copy()
        return base.assign(n_fires=0)

    # Ensure common CRS
    if peru_lvl2_gdf.crs is None:
        raise ValueError("Peru polygons have no CRS. Set to EPSG:4326 or appropriate CRS.")
    if fires_gdf.crs is None:
        raise ValueError("fires_gdf has no CRS. Ensure `fires_clean` was used or set CRS.")

    if peru_lvl2_gdf.crs.to_string() != fires_gdf.crs.to_string():
        fires = fires_gdf.to_crs(peru_lvl2_gdf.crs)
    else:
        fires = fires_gdf

    # Spatial join: point within polygon
    joined = gpd.sjoin(fires, peru_lvl2_gdf[[id_col, peru_lvl2_gdf.geometry.name] + ([] if not keep_province_attrs else keep_province_attrs)],
                       how="inner", predicate="within")

    # Grouping keys
    keys = [id_col]
    if level == "year_month":
        keys += ["year", "month"]
    else:
        keys += ["year"]

    # Count fires + extra aggregations
    agg_dict = {"geometry": "count"}  # we will rename to n_fires
    if agg_extra:
        agg_dict.update(agg_extra)

    grp = joined.groupby(keys, dropna=False).agg(agg_dict).reset_index()
    grp = grp.rename(columns={"geometry": "n_fires"})

    # Optionally attach province attrs (deduplicate first)
    if keep_province_attrs:
        attrs = joined[[id_col] + keep_province_attrs].drop_duplicates(subset=[id_col])
        grp = grp.merge(attrs, on=id_col, how="left")

    # Ensure ints for time keys
    if "year" in grp.columns:
        grp["year"] = grp["year"].astype("Int64")
    if "month" in grp.columns:
        grp["month"] = grp["month"].astype("Int64")

    # Sort nicely
    sort_cols = [c for c in [id_col, "year", "month"] if c in grp.columns]
    grp = grp.sort_values(sort_cols).reset_index(drop=True)

    grpd = gpd.GeoDataFrame(peru_lvl2_gdf.merge(grp, on = id_col))
    return grpd


In [37]:
import os
import io
import zipfile
from datetime import date
from typing import List, Optional, Tuple

import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium

In [49]:
# 5) Folium choropleth
def folium_choropleth_peru(
    prov_counts_gdf: gpd.GeoDataFrame,
    value_col: str = "fire_count",
    prov_id: str = "fire_count",
    name_col: str = "province",
    out_html: str = "peru_viirs_fires.html",
) -> folium.Map:
    """
    Builds and saves a Folium choropleth map. Returns the map object.
    """
    # Center roughly on Peru
    m = folium.Map(location=[-9.2, -75.0], zoom_start=5, tiles="cartodbpositron")

    # Prepare choropleth
    chor = folium.Choropleth(
        geo_data=prov_counts_gdf.to_json(),
        data=prov_counts_gdf,
        columns=[prov_id, value_col],
        key_on=f"feature.properties.{prov_id}",
        fill_opacity=0.8,
        line_opacity=0.7,
        legend_name="VIIRS active fires (count)",
        highlight=True,
        nan_fill_opacity=0.2,
        bins=8,
    )
    chor.add_to(m)

    # Add interactive tooltip
    folium.GeoJson(
        prov_counts_gdf.to_json(),
        name="Provinces",
        style_function=lambda x: {"weight": 0.5, "color": "#555", "fillOpacity": 0},
        tooltip=folium.GeoJsonTooltip(
            fields=[name_col, value_col],
            aliases=["Province", "Fires"],
            localize=True,
            sticky=False,
        ),
    ).add_to(m)

    folium.LayerControl().add_to(m)
    m.save(out_html)
    return m

In [57]:
def download_ctry_shp( iso3 = "PER"):

    ctry = gpd.read_file(r"https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_PER_1.json")
    return ctry

In [12]:
import geopandas as gpd

In [58]:
ctry = download_ctry_shp(iso3="PER")

In [8]:
fires = download_viirs_snpp_peru(year = 2015)

In [ ]:
gdf_fires = fires_clean(fires)

C:\Users\187697\AppData\Local\Temp\ipykernel_25940\3936079418.py:39: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  dt = pd.to_datetime(df[acq_time_col], errors="coerce", utc=False, infer_datetime_format=True)


In [59]:
int_year = intersect_and_collapse_fires_peru(
    gdf_fires, ctry, id_col='GID_1', level="year", agg_extra={"frp": "sum"},
    keep_province_attrs=["NAME_1"]  # optional nice labels
)

In [60]:
 _map = folium_choropleth_peru(int_year, value_col="n_fires", name_col="NAME_1_x",prov_id='GID_1',
                                  out_html=r"C:\Users\187697\Downloads/peru_viirs_fires.html")